# MultiLogiEval d5 Two-Stage: Error Classification Analysis

Comprehensive analysis of 27 verified-but-wrong cases at depth-5 (hardest reasoning) using error root cause classification.

**Depth-5 Context**: d5 represents the hardest reasoning tasks with 5 chained logical rules, requiring the model to track and combine multiple inference steps to reach the correct conclusion.

**Two-Stage Approach**: Natural language reasoning followed by Lean formalization and verification.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define consistent color palette
ERROR_COLORS = {
    'AXIOMATIZES_CONCLUSION': '#e74c3c',
    'AXIOMATIZES_CONTRADICTION': '#3498db',
    'INCORRECT_FORMALIZATION': '#f39c12',
    'REASONING_FAILURE': '#2ecc71',
    'AXIOMATIZES_UNMENTIONED': '#9b59b6',
    'OTHER': '#95a5a6'
}

## 1. Load Data and Basic Statistics

In [ ]:
# Load the d5-only error analysis CSV
df_d5 = pd.read_csv('../results/multilogieval/d5_only/two_stage/error_root_cause_analysis.csv')

print(f"Total verified-but-wrong cases at d5: {len(df_d5)}")
print(f"\nColumns: {list(df_d5.columns)}")
print(f"\nData shape: {df_d5.shape}")
print(f"\nUnique error types: {df_d5['Root_Cause_Category'].nunique()}")
print(f"\nUnique patterns: {df_d5['Pattern'].nunique()}")
print(f"\nLogic types: {df_d5['Logic_Type'].unique()}")
print(f"\nDepth levels: {sorted(df_d5['Depth'].unique())}")

In [ ]:
# Show first few rows
df_d5.head()

## 2. Error Type Distribution

In [ ]:
# Calculate error distribution
error_dist = df_d5['Root_Cause_Category'].value_counts()

print("Root Cause Distribution (d5 only):")
print("=" * 70)
for category, count in error_dist.items():
    percentage = (count / len(df_d5)) * 100
    print(f"{category:30s} {count:3d} cases ({percentage:5.1f}%)")
print(f"\nTotal: {len(df_d5)} cases")

# Calculate axiomatization total
axiom_types = ['AXIOMATIZES_CONCLUSION', 'AXIOMATIZES_CONTRADICTION', 'AXIOMATIZES_UNMENTIONED']
axiom_total = sum(error_dist.get(t, 0) for t in axiom_types)
print(f"\nTotal Axiomatization Errors: {axiom_total} ({axiom_total/len(df_d5)*100:.1f}%)")

In [ ]:
# Visualize error distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
error_dist.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_title('Error Type Distribution at d5 (Bar Chart)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Error Category', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(error_dist.values):
    ax1.text(i, v + 0.3, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
colors_pie = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
ax2.pie(error_dist.values, labels=error_dist.index, autopct='%1.1f%%', 
        colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Error Type Distribution at d5 (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Error Types by Logic Type (FOL/NM/PL)

In [ ]:
# Error distribution by logic type
logic_error = pd.crosstab(df_d5['Logic_Type'], df_d5['Root_Cause_Category'])

print("Error Types by Logic Type (d5):")
print("=" * 70)
print(logic_error)

print("\nBreakdown by Logic Type:")
for logic in sorted(df_d5['Logic_Type'].unique()):
    logic_count = len(df_d5[df_d5['Logic_Type'] == logic])
    if logic_count > 0:
        top_err_counts = df_d5[df_d5['Logic_Type'] == logic]['Root_Cause_Category'].value_counts()
        top_err_name = top_err_counts.index[0]
        top_err = top_err_counts.iloc[0]
        print(f"  {logic}: {logic_count} cases, most common: {top_err_name} ({top_err})")

In [ ]:
# Visualize logic type distribution
fig, ax = plt.subplots(figsize=(14, 6))

# By logic type
ordered_cols = error_dist.index.tolist()
logic_error_ordered = logic_error[ordered_cols]
logic_error_ordered.plot(kind='bar', ax=ax,
                         color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                         edgecolor='black', linewidth=1.2)
ax.set_title('Error Types by Logic Type (d5)', fontsize=14, fontweight='bold')
ax.set_xlabel('Logic Type', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparison with All-Depths Error Patterns

In [ ]:
# Load all-depths data for comparison
df_all = pd.read_csv('../results/multilogieval/all_depths/two_stage/error_root_cause_analysis.csv')

print(f"All-depths verified-but-wrong cases: {len(df_all)}")
print(f"d5-only verified-but-wrong cases: {len(df_d5)}")
print(f"d5 represents {len(df_d5)/len(df_all)*100:.1f}% of all errors")

In [ ]:
# Compare error distributions
error_dist_all = df_all['Root_Cause_Category'].value_counts(normalize=True) * 100
error_dist_d5_pct = df_d5['Root_Cause_Category'].value_counts(normalize=True) * 100

# Create comparison dataframe
all_errors = sorted(set(error_dist_all.index) | set(error_dist_d5_pct.index))
comparison = pd.DataFrame({
    'All Depths (%)': [error_dist_all.get(e, 0) for e in all_errors],
    'd5 Only (%)': [error_dist_d5_pct.get(e, 0) for e in all_errors]
}, index=all_errors)

print("\nError Distribution Comparison:")
print("=" * 70)
print(comparison.round(1))
print("\nDifference (d5 - All):")
comparison['Difference'] = comparison['d5 Only (%)'] - comparison['All Depths (%)']
print(comparison[['Difference']].round(1).sort_values('Difference', ascending=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(14, 7))

x = range(len(comparison))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], comparison['All Depths (%)'], 
               width, label='All Depths', color='#95a5a6', edgecolor='black', linewidth=1.2)
bars2 = ax.bar([i + width/2 for i in x], comparison['d5 Only (%)'], 
               width, label='d5 Only', color='#e74c3c', edgecolor='black', linewidth=1.2)

ax.set_xlabel('Error Type', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Error Distribution: d5 vs All Depths (Two-Stage)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Key Findings

In [ ]:
print("=" * 70)
print("KEY FINDINGS: MULTILOGIEVAL d5 TWO-STAGE ERROR CLASSIFICATION")
print("=" * 70)

# Top 3 error types
top3 = error_dist.head(3)
print("\n1. TOP 3 ERROR TYPES AT d5:")
for i, (cat, count) in enumerate(top3.items(), 1):
    pct = count / len(df_d5) * 100
    print(f"   #{i}: {cat}")
    print(f"       {count} cases ({pct:.1f}%)")

print(f"\n2. AXIOMATIZATION ERRORS AT d5:")
print(f"   Total: {axiom_total}/{len(df_d5)} ({axiom_total/len(df_d5)*100:.1f}%)")
if axiom_total > 0:
    print(f"   Model axiomatizes instead of proving in {axiom_total}/{len(df_d5)} cases")

print(f"\n3. LOGIC TYPE BREAKDOWN AT d5:")
for logic in sorted(df_d5['Logic_Type'].unique()):
    logic_count = len(df_d5[df_d5['Logic_Type'] == logic])
    if logic_count > 0:
        top_err_counts = df_d5[df_d5['Logic_Type'] == logic]['Root_Cause_Category'].value_counts()
        top_err = top_err_counts.iloc[0]
        top_err_name = top_err_counts.index[0]
        print(f"   {logic}: {logic_count} cases, most common: {top_err_name} ({top_err})")

print(f"\n4. WHY d5 IS HARDER:")
print(f"   - Depth-5 requires chaining 5 logical rules to reach conclusion")
print(f"   - Longer reasoning chains increase formalization complexity")
print(f"   - Two-stage approach: NL reasoning → Lean formalization")
incorrect_form = error_dist.get('INCORRECT_FORMALIZATION', 0)
print(f"   - INCORRECT_FORMALIZATION rate: {incorrect_form/len(df_d5)*100:.1f}% at d5")
reasoning_fail = error_dist.get('REASONING_FAILURE', 0)
print(f"   - REASONING_FAILURE rate: {reasoning_fail/len(df_d5)*100:.1f}% at d5")

# Calculate rates for all depths
incorrect_all = df_all['Root_Cause_Category'].value_counts(normalize=True).get('INCORRECT_FORMALIZATION', 0) * 100
reasoning_all = df_all['Root_Cause_Category'].value_counts(normalize=True).get('REASONING_FAILURE', 0) * 100
print(f"   - Comparison: {incorrect_all:.1f}% / {reasoning_all:.1f}% across all depths")

print(f"\n5. d5 vs ALL-DEPTHS COMPARISON:")
print(f"   - d5 accounts for {len(df_d5)}/{len(df_all)} ({len(df_d5)/len(df_all)*100:.1f}%) of total errors")
print(f"   - Biggest differences in error patterns:")
for idx, row in comparison.sort_values('Difference', ascending=False).head(3).iterrows():
    diff_str = f"+{row['Difference']:.1f}%" if row['Difference'] > 0 else f"{row['Difference']:.1f}%"
    print(f"     {idx}: {diff_str} at d5")

print("\n" + "=" * 70)
print("CONCLUSION:")
print("=" * 70)
print(f"\nDepth-5 represents the hardest reasoning tasks with 5 chained rules.")
print(f"The {len(df_d5)} verified-but-wrong cases in two-stage approach show")
print(f"that INCORRECT_FORMALIZATION ({incorrect_form/len(df_d5)*100:.1f}%) dominates,")
print(f"indicating that translating complex NL reasoning to Lean is the main")
print(f"bottleneck at depth-5, rather than the reasoning itself.")
print("=" * 70)

## 6. Example Cases from Top Error Types

In [ ]:
# Get top 3 error types
top_3_types = error_dist.head(3).index.tolist()

for error_type in top_3_types:
    subset = df_d5[df_d5['Root_Cause_Category'] == error_type]
    count = len(subset)
    pct = count / len(df_d5) * 100
    
    print("\n" + "=" * 70)
    print(f"{error_type} ({count} cases, {pct:.1f}%)")
    print("=" * 70)
    
    # Show up to 3 examples
    for idx, (_, row) in enumerate(subset.head(3).iterrows(), 1):
        print(f"\nExample {idx}:")
        print(f"  Logic: {row['Logic_Type']}, Depth: {row['Depth']}")
        print(f"  Pattern: {row['Pattern']}")
        print(f"  Context: {str(row['Context'])[:120]}...")
        print(f"  Question: {str(row['Question'])[:120]}...")
        if pd.notna(row['Problematic_Lines']):
            print(f"  Problematic Lines: {row['Problematic_Lines']}")
        print(f"  Error: {str(row['Error_Description'])[:150]}...")